# 🏔️ Fase 1: Exploración y Hallazgos
## Detección de Deslizamientos de Tierra con Machine Learning
**Asignatura:** Visualización de Datos · 2026  
**Dataset:** Landslide4Sense — Sentinel-1/2 · ALOS DEM · 14 canales · 3799 muestras  
**Objetivo:** Descubrir qué modelos y qué señales del terreno son más discriminativas para detectar deslizamientos.

---

## 1. Pregunta de Negocio

> **¿Qué modelo de Machine Learning detecta mejor los deslizamientos de tierra en imágenes satelitales multiespectrales, y qué características físicas del terreno son más determinantes para la predicción?**

**Contexto:** Los deslizamientos de tierra causan miles de muertes y miles de millones de dólares en pérdidas anuales. Detectarlos automáticamente desde imágenes satelitales permitiría alertas tempranas y mapeo rápido de riesgo. El dataset Landslide4Sense contiene imágenes de 14 bandas (ópticas, SAR, topográficas) etiquetadas como landslide (positivo) o no-landslide (negativo).

**Modelos evaluados:**
- **Clásicos:** Logistic Regression, SVM (RBF), Random Forest
- **Deep Learning:** ResNet-50, EfficientNet-B4, U-Net ResNet-34

**Métricas clave:** F1 Score (balance precisión-recall), AUC-ROC, Recall (prioridad en alertas tempranas)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Estilo básico exploratorio (sin pulir)
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('Librerías cargadas ✓')

In [ ]:
# ── Cargar datos de resultados ────────────────────────────────────────────────
df_models = pd.read_csv('../data/comparison_table.csv')
df_channels = pd.read_csv('../data/channel_stats_by_class.csv')

with open('../data/final_summary.json') as f:
    summary = json.load(f)

print('Datos cargados:')
print(f'  - Modelos evaluados: {len(df_models)}')
print(f'  - Canales analizados: {len(df_channels)}')
print(f'  - Mejor modelo (resumen): {summary["best_model"]} (F1={summary["best_f1"]:.4f})')
print()
df_models

---
## 2. Exploración 1: ¿Qué modelo tiene mayor F1 Score?

**Hipótesis inicial:** Esperamos que los modelos de Deep Learning superen a los clásicos, dado su mayor capacidad de representación.

Comenzamos con una visualización exploratoria directa: barras simples de F1 medio por modelo.

In [ ]:
# ── Exploración 1: Barras de F1 por modelo (estilo exploratorio raw) ──────────
fig, ax = plt.subplots(figsize=(10, 5))

df_sorted = df_models.sort_values('F1 medio', ascending=False)
colors = ['#2ca02c' if t == 'Clásico' else '#9467bd' for t in df_sorted['Tipo']]

bars = ax.bar(df_sorted['Modelo'], df_sorted['F1 medio'],
              color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)

# Barras de error (std)
ax.errorbar(df_sorted['Modelo'], df_sorted['F1 medio'],
            yerr=df_sorted['Std'].fillna(0),
            fmt='none', color='#333333', capsize=5, linewidth=1.5)

# Valores sobre las barras
for bar, val in zip(bars, df_sorted['F1 medio']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='black')

ax.set_ylim(0, 1.0)
ax.set_xlabel('Modelo')
ax.set_ylabel('F1 Score Medio (5-fold CV)')
ax.set_title('F1 Score por Modelo — Detección de Deslizamientos (exploración inicial)')
ax.tick_params(axis='x', rotation=20)

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ca02c', label='Clásico'),
                   Patch(facecolor='#9467bd', label='Deep Learning')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig('../data/figures/exploracion_1_f1_barras.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: exploracion_1_f1_barras.png')

### 📝 Análisis Exploración 1

**Observación inesperada:** El modelo **Random Forest (F1=0.837)** supera a todos los modelos de Deep Learning, incluyendo ResNet-50 (F1=0.784) y EfficientNet-B4 (F1=0.755). U-Net ResNet-34 —diseñada específicamente para segmentación semántica— obtiene el peor resultado (F1=0.444).

**¿Por qué?** Exploramos más en la siguiente visualización...

In [ ]:
# ── Análisis complementario: Precisión vs Recall (scatter exploratorio) ───────
df_completo = df_models.dropna(subset=['Precisión', 'Recall']).copy()

fig, ax = plt.subplots(figsize=(8, 6))

scatter_colors = {'Clásico': '#2ca02c', 'Deep Learning': '#9467bd'}
for tipo, grupo in df_completo.groupby('Tipo'):
    ax.scatter(grupo['Precisión'], grupo['Recall'],
               c=scatter_colors[tipo], s=grupo['F1 medio']*300,
               label=tipo, alpha=0.85, edgecolors='white', linewidth=1)
    for _, row in grupo.iterrows():
        ax.annotate(row['Modelo'], (row['Precisión'], row['Recall']),
                    textcoords='offset points', xytext=(8, 5), fontsize=9)

# Curvas iso-F1
for f1 in [0.75, 0.80, 0.85]:
    prec = np.linspace(0.01, 1.0, 300)
    rec = f1 * prec / (2 * prec - f1 + 1e-9)
    mask = (rec > 0) & (rec <= 1) & (prec <= 1)
    ax.plot(prec[mask], rec[mask], '--', color='gray', alpha=0.5, linewidth=1)
    # Etiquetar la curva
    idx = len(prec[mask])//2
    ax.text(prec[mask][idx]+0.01, rec[mask][idx], f'F1={f1}',
            fontsize=8, color='gray', alpha=0.7)

ax.set_xlabel('Precisión')
ax.set_ylabel('Recall')
ax.set_title('Trade-off Precisión vs Recall — Modelos Clásicos\n(tamaño burbuja ∝ F1 Score)')
ax.legend()
ax.set_xlim(0.65, 0.90)
ax.set_ylim(0.70, 1.02)
plt.tight_layout()
plt.savefig('../data/figures/exploracion_prec_recall.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Exploración 2: ¿Qué canales espectrales discriminan mejor los deslizamientos?

**Pregunta:** ¿Existe diferencia estadística entre la señal de píxeles de deslizamiento (positivos) vs. no-deslizamiento (negativos) en cada canal?

Calculamos el **delta** (diferencia de medias) como proxy de poder discriminativo.

In [ ]:
# ── Exploración 2: Delta de canales (exploración de señales) ──────────────────
df_ch = df_channels.copy()
df_ch['|Delta|'] = df_ch['Delta'].abs()
df_ch = df_ch.sort_values('|Delta|', ascending=True)

# Categorías de sensor
def sensor_cat(name):
    if 'SAR' in name or 'VV' in name or 'VH' in name: return 'SAR'
    elif 'DEM' in name or 'Slope' in name: return 'Topografía'
    elif 'RedEdge' in name: return 'RedEdge'
    else: return 'Óptico'

df_ch['Sensor'] = df_ch['Nombre'].apply(sensor_cat)
palette = {'SAR': '#F59E0B', 'Topografía': '#EF4444', 'RedEdge': '#D62728', 'Óptico': '#3B82F6'}

fig, ax = plt.subplots(figsize=(10, 7))

colors_bars = [palette[s] for s in df_ch['Sensor']]
bars = ax.barh(df_ch['Nombre'], df_ch['|Delta|'], color=colors_bars, alpha=0.85, edgecolor='white')

# Valores
for bar, val in zip(bars, df_ch['|Delta|']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('|Δ media| = |Media_Landslide − Media_NoLandslide|')
ax.set_title('Poder Discriminativo por Canal Espectral\n(exploración — mayor Δ = más informativo)')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in palette.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../data/figures/exploracion_2_canales_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop 5 canales más discriminativos:')
print(df_ch.sort_values('|Delta|', ascending=False)[['Nombre', 'Sensor', '|Delta|', 'Delta']].head(5).to_string(index=False))

### 📝 Análisis Exploración 2

**Patrón claro:** Los canales **RedEdge3 (B7, |Δ|=0.807)** y **RedEdge2 (B6, |Δ|=0.563)** sobresalen notablemente sobre el resto. Estos canales de borde rojo de Sentinel-2 son altamente sensibles a la vegetación y a suelos expuestos —exactamente lo que caracteriza una zona de deslizamiento fresco.

Los canales ópticos convencionales (Azul, Verde, Rojo, NIR) muestran poca diferencia entre clases, lo que explica por qué los modelos que priorizan esos canales tienen menor rendimiento.

In [ ]:
# ── Exploración 3: Comparación media por clase (top 6 canales) ──────────────
df_top6 = df_channels.sort_values('Delta', key=abs, ascending=False).head(6)

x = np.arange(len(df_top6))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(x - width/2, df_top6['Media_Pos'], width,
               label='Landslide (Positivo)', color='#D62728', alpha=0.8,
               yerr=df_top6['Std_Pos'], capsize=4)
bars2 = ax.bar(x + width/2, df_top6['Media_Neg'], width,
               label='No-Landslide (Negativo)', color='#1F77B4', alpha=0.8,
               yerr=df_top6['Std_Neg'], capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(df_top6['Nombre'], rotation=15, ha='right')
ax.set_ylabel('Media de intensidad normalizada')
ax.set_title('Media por Clase — Top 6 Canales Más Discriminativos')
ax.legend()

plt.tight_layout()
plt.savefig('../data/figures/exploracion_3_medias_clase.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Exploración 4: Distribución de F1 y varianza por modelo ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfica 1: F1 medio con intervalo de confianza aproximado
df_sorted = df_models.sort_values('F1 medio', ascending=False).reset_index(drop=True)
colors_list = ['#D62728' if i == 0 else ('#2ca02c' if t == 'Clásico' else '#9467bd')
               for i, t in enumerate(df_sorted['Tipo'])]

axes[0].barh(df_sorted['Modelo'][::-1], df_sorted['F1 medio'][::-1],
             color=colors_list[::-1], alpha=0.85)
axes[0].errorbar(df_sorted['F1 medio'][::-1], df_sorted['Modelo'][::-1],
                 xerr=df_sorted['Std'].fillna(0)[::-1],
                 fmt='none', color='black', capsize=4)
axes[0].axvline(x=0.8, color='gray', linestyle='--', alpha=0.7, label='F1 = 0.80')
axes[0].set_xlabel('F1 Score Medio')
axes[0].set_title('Ranking de Modelos (media ± std)')
axes[0].legend(fontsize=9)

# Gráfica 2: Recall — relevante para alertas tempranas
df_rec = df_models.dropna(subset=['Recall']).sort_values('Recall', ascending=True)
colors_rec = ['#D62728' if m == 'Random Forest' else '#7F7F7F' for m in df_rec['Modelo']]
axes[1].barh(df_rec['Modelo'], df_rec['Recall'], color=colors_rec, alpha=0.85)
axes[1].axvline(x=0.9, color='#D62728', linestyle='--', alpha=0.7, label='Recall = 0.90')
axes[1].set_xlabel('Recall')
axes[1].set_title('Recall por Modelo\n(crítico para alertas tempranas)')
axes[1].set_xlim(0.7, 1.0)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../data/figures/exploracion_4_recall.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. 🎯 El Hallazgo

Después del análisis exploratorio, podemos articular dos hallazgos principales:

---

### Hallazgo 1: Random Forest supera a las redes neuronales profundas

**Anomalía encontrada:** Contra la intuición dominante en computer vision, el modelo **Random Forest (F1=0.837)** supera a todos los modelos de Deep Learning. Esto se explica porque:
- El dataset Landslide4Sense (3799 muestras) es **relativamente pequeño** para entrenar redes profundas desde cero
- Las features engineered (14 canales ya pre-procesados) dan ventaja a los métodos de ensamble sobre las CNN que deben aprender representaciones
- U-Net (F1=0.444) sufre especialmente por su alta capacidad paramétrica sin suficientes datos de entrenamiento

**Implicación de negocio:** Para sistemas de detección con datasets limitados y features bien definidas, los modelos clásicos de ML son la primera elección, no las arquitecturas profundas.

---

### Hallazgo 2: Los canales RedEdge (B6, B7) son la señal más discriminativa

**Correlación descubierta:** Las bandas espectrales de **borde rojo** (RedEdge) de Sentinel-2 muestran una diferencia de medias (Δ) hasta **10 veces mayor** que los canales ópticos convencionales (Azul, Verde, Rojo). Esta señal corresponde físicamente a la respuesta espectral del suelo desnudo expuesto durante un deslizamiento.

**Implicación de negocio:** En sistemas de alerta temprana, priorizar imágenes con bandas RedEdge disponibles (Sentinel-2 Nivel 2A) maximizará la precisión de detección. Adicionalmente, el **DEM de elevación (Δ=0.195)** y **SAR-VH (Δ=0.188)** complementan la señal óptica con información topográfica y de rugosidad de superficie.

---

**Estos hallazgos guiarán el diseño del Dashboard aclaratorio (Fase 2).**

In [ ]:
# ── Resumen cuantitativo del hallazgo ─────────────────────────────────────────
print('='*60)
print('RESUMEN DE HALLAZGOS — EXPLORACIÓN')
print('='*60)
print()
print('Hallazgo 1: Ranking de modelos por F1 Score')
for _, row in df_models.sort_values('F1 medio', ascending=False).iterrows():
    marker = '⭐' if row['Modelo'] == 'Random Forest' else '  '
    print(f'  {marker} {row["Modelo"]:<25} F1={row["F1 medio"]:.4f}  [{row["Tipo"]}]')

print()
print('Hallazgo 2: Top 5 canales más discriminativos (|Δ media|)')
df_channels_sorted = df_channels.copy()
df_channels_sorted['|Delta|'] = df_channels_sorted['Delta'].abs()
for _, row in df_channels_sorted.sort_values('|Delta|', ascending=False).head(5).iterrows():
    print(f'  Canal {int(row["Canal"]):2d} — {row["Nombre"]:<25} |Δ|={row["|Delta|"]:.4f}')

print()
print('→ Estos hallazgos serán el centro del Dashboard aclaratorio.')
print('='*60)